In [1]:
!pip install pandas matplotlib folium


[notice] A new release of pip is available: 24.1 -> 24.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import folium

In [7]:
industry_data = pd.read_csv('23년_광주_산업별_월별_판매량.csv', encoding='utf-8')  # 산업별 전력 판매량 데이터
population_data = pd.read_excel('광주광역시 23 유동인구.xlsx')  # 유동인구 데이터

In [8]:
# 산업별로 전력 소비량을 요약하여 빈집 가능성이 높은 지역 분석
industry_data_grouped = industry_data.groupby(['시군구', '읍면동(법정동)']).agg({
    '판매량': 'sum'  # 전력 소비량 합계
}).reset_index()

In [9]:
# 유동인구 데이터 요약
population_data_grouped = population_data.groupby('지역').agg({
    '전체': 'sum'  # 총 인구 수
}).reset_index()

In [10]:
# 빈집 가능성이 높은 지역을 선정하기 위해 전력 소비량 대비 인구 비율 계산
merged_data = pd.merge(industry_data_grouped, population_data_grouped, left_on='시군구', right_on='지역', how='inner')
merged_data['power_to_population_ratio'] = merged_data['판매량'] / merged_data['전체']

# 빈집 가능성이 높은 상위 3개 지역 선정
top_3_low_activity_areas = merged_data.nsmallest(3, 'power_to_population_ratio')

In [11]:
# 위도와 경도를 기반으로 해당 지역을 지도에 시각화
top_3_low_activity_areas['위도'] = [35.13995836, 35.14895836, 35.15895836]  # 랜덤 데이터
top_3_low_activity_areas['경도'] = [126.793668, 126.803668, 126.813668]  # 랜덤 데이터

# 지도 생성
m = folium.Map(location=[35.1595454, 126.8526012], zoom_start=12)  # 광주광역시 중심

# 각 지역을 지도에 마커로 표시
for _, row in top_3_low_activity_areas.iterrows():
    folium.Marker([row['위도'], row['경도']], popup=f"{row['시군구']} {row['읍면동(법정동)']}").add_to(m)

# 지도 출력
m.save("top_3_areas_map.html")

In [12]:
# 빈집 가능성이 높은 지역에서 낮은 전력 소비량을 보이는 산업을 추출합니다.
top_3_areas_industry = industry_data[industry_data['읍면동(법정동)'].isin(top_3_low_activity_areas['읍면동(법정동)'])]
industry_summary = top_3_areas_industry.groupby('산업분류명(대)').agg({
    '판매량': 'sum'  # 산업별 전력 소비량 합계
}).reset_index()

# 전력 소비량이 낮은 산업을 기반으로 추출
underdeveloped_industries = industry_summary.sort_values(by='판매량').head()

In [13]:
# 각 상위 3개 지역에 전력 소비량이 낮은 산업 추출
optimized_industry_placement_manual = pd.DataFrame({
    '시군구': ['광산구', '북구', '광산구'],
    '읍면동(법정동)': ['도호동', '덕의동', '명도동'],
    '추천 산업': ['도매 및 소매업', '사업시설 관리, 사업 지원 및 임대 서비스업', '부동산업']
})

In [14]:
# 결과 출력
print(optimized_industry_placement_manual)

   시군구 읍면동(법정동)                     추천 산업
0  광산구      도호동                  도매 및 소매업
1   북구      덕의동  사업시설 관리, 사업 지원 및 임대 서비스업
2  광산구      명도동                      부동산업


In [15]:
# 결과 지도 시각화 출력 (folium으로 생성된 지도)
m